# Day 4: Exploratory Data Analysis & Market Dynamics
**Prepared by:** Aditya (Lead), Anhum (Visualizations), Tanveer (Statistical Findings)

In this notebook we conduct a rigorous empirical investigation of:
1. **Return Distribution Properties**: Leptokurtosis, negative skewness, non-normality
2. **ARCH Volatility Clustering**: Empirical evidence of time-varying conditional variance
3. **Cross-Asset Correlation Dynamics**: Diversification potential of our investment universe
4. **Drawdown Anatomy**: Peak-to-trough loss profiles and recovery timelines
5. **Rolling Market Statistics**: Beta drift, Sharpe variability, regime shifts

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from src.utils.finance_math import calculate_log_returns, calculate_arithmetic_returns
from src.utils.eda_engine import DistributionAnalyzer, RollingMarketMetrics, DrawdownAnalyzer, CrossAssetAnalyzer

symbols = ['AAPL', 'MSFT', 'NVDA', 'SPY', 'QQQ']
dfs = {sym: pd.read_parquet(f'../data/processed/{sym}_processed.parquet') for sym in symbols}
log_rets = {sym: calculate_log_returns(dfs[sym]['Adj Close']) for sym in symbols}
arith_rets = {sym: calculate_arithmetic_returns(dfs[sym]['Adj Close']) for sym in symbols}
print('Data loaded successfully for', list(dfs.keys()))

## 1. Distributional Statistics — Leptokurtosis Evidence

In [ ]:
records = [DistributionAnalyzer.describe_returns(log_rets[sym], sym) for sym in symbols]
pd.DataFrame(records).set_index('label')[['ann_return','ann_vol','skewness','kurtosis','jarque_bera_pval','rejects_normality']].round(4)

## 2. Volatility Clustering: Rolling Volatility of AAPL

In [ ]:
ret = log_rets['AAPL']
rolling_vol = RollingMarketMetrics.rolling_volatility(ret, window=21)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].bar(ret.index, ret.values, color='grey', alpha=0.5, width=1)
axes[0].set_title('AAPL Daily Returns & Rolling Volatility — ARCH Clustering', fontweight='bold')
axes[0].set_ylabel('Daily Log Return')
axes[1].fill_between(rolling_vol.index, rolling_vol.values, alpha=0.4, color='crimson')
axes[1].plot(rolling_vol.index, rolling_vol.values, color='crimson', lw=1.5, label='21d Rolling Annualized Vol')
axes[1].set_ylabel('Annualized Volatility')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Cross-Asset Correlation Heatmap

In [ ]:
corr = CrossAssetAnalyzer.correlation_matrix(log_rets)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Cross-Asset Log Return Correlation (2018–2024)', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Average Pairwise Correlation: {CrossAssetAnalyzer.average_pairwise_correlation(corr):.3f}')

## 4. Underwater Drawdown Curves

In [ ]:
palette = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#9467bd']
fig, ax = plt.subplots(figsize=(12, 5))
for idx, sym in enumerate(symbols):
    wealth = (1 + arith_rets[sym]).cumprod()
    dd = DrawdownAnalyzer.compute_drawdown_series(wealth)
    profile = DrawdownAnalyzer.full_drawdown_profile(arith_rets[sym])
    ax.fill_between(dd.index, dd.values * 100, 0, alpha=0.2, color=palette[idx])
    ax.plot(dd.index, dd.values * 100, color=palette[idx], lw=1.2,
            label=f'{sym} (MDD={profile["max_drawdown_pct"]}, {profile["max_drawdown_duration_days"]}d)')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Underwater Drawdown Curves — All Assets', fontweight='bold', fontsize=12)
ax.set_ylabel('Drawdown (%)')
ax.legend(fontsize=8, frameon=True)
ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

## 5. Rolling Beta: NVDA vs SPY

In [ ]:
rolling_beta = RollingMarketMetrics.rolling_beta(log_rets['NVDA'], log_rets['SPY'], window=63)
rolling_sharpe = RollingMarketMetrics.rolling_sharpe(log_rets['NVDA'], window=63)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(rolling_beta.index, rolling_beta.values, color='#1f77b4', lw=1.5)
axes[0].axhline(1.0, color='grey', ls='--', lw=0.8, label='Beta = 1')
axes[0].set_title('NVDA: 63-Day Rolling Beta (vs SPY) & Rolling Sharpe', fontweight='bold')
axes[0].set_ylabel('Beta (Market Sensitivity)')
axes[0].legend()
axes[1].plot(rolling_sharpe.index, rolling_sharpe.values, color='#2ca02c', lw=1.5)
axes[1].axhline(0, color='black', lw=0.5, ls='--')
axes[1].set_ylabel('Annualized Sharpe Ratio')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()